**Legacy notebook.** Self-contained analysis code that predates the `src/mrvf` library and has not been ported to it. Kept for provenance and because it still produces figures in `results/`. Paths were updated to the `results/` layout; the next cell sets the working directory to the repository root, so run it from anywhere.

For the maintained pipeline see `notebooks/01_train_triple_regime.ipynb` and `notebooks/02_evaluate_rmse_vs_snr.ipynb`.

In [ ]:
import os
from pathlib import Path
# run from the repository root so ./results/... and ../subsamples resolve
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "mrvf").is_dir())
os.chdir(_root)

# Per-Subject Voxel Distributions — Triple (A+B+C) DL vs DM

Reproduces the reference figure style: one histogram line per subject,
bold group-mean curve on top, shaded literature range.

**Source**: smoothed `.mat` files from `InVivo_SmoothFirst_v1.ipynb`  
**Mask**: GM voxels only (label == 2 from `_AIR_ROI.nii.gz`)  
**Condition**: air / baseline only

**Literature ranges used:**

| Parameter | Range | Key Reference |
|-----------|-------|---------------|
| SO₂ | 58–70 % | Christen et al. *NeuroImage* 2014; Christen et al. *MRM* 2012 |
| CBV | 3–6 % | Christen et al. *NeuroImage* 2014 (3.1±0.7%); Li et al. *MRM* 2021 (5.4±0.6%) |
| R | 8–15 µm | Christen et al. *NeuroImage* 2014 (12.6±2.4 µm) |
| T2 | 60–90 ms | Ni et al. *MRM* 2015; standard GM T2 at 3T |

## 1. Imports & style

In [ ]:
import os, glob, re
import numpy as np
import scipy.io as sio
import nibabel as nib
import h5py
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.family'    : 'Arial',
    'font.size'      : 9,
    'axes.labelsize' : 10,
    'axes.titlesize' : 11,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi'     : 150,
    'savefig.dpi'    : 300,
    'savefig.bbox'   : 'tight',
})

C_DL  = '#1565C0'   # deep blue  — Triple (A+B+C)
C_DM  = '#E65100'   # deep orange — DM

print('Imports OK')

## 2. Configuration — **edit paths here**

In [ ]:
CONFIG = {
    # ── Smoothed map files from InVivo_SmoothFirst_v1 ─────────────────────
    'maps_dir'    : './results/invivo_smooth_v1',
    'maps_pattern': '*_maps_smooth.mat',

    # ── GM mask files (same dir as used throughout) ───────────────────────
    'masks_dir'   : '../GESFIDE_data/GES_ROI',

    # ── Output ────────────────────────────────────────────────────────────
    'output_dir'  : './results/invivo_smooth_v1/figures',

    # ── Histogram bins ────────────────────────────────────────────────────
    'n_bins'      : 80,

    # ── Literature ranges [lo, hi] — adjust freely ────────────────────────
    # SO2 (%):  Christen 2014 (59.5+/-4.7%); Christen 2012 (60+/-6%)
    # CBV (%):  Christen 2014 (3.1+/-0.7%);  Li 2021 (5.4+/-0.6%)
    # R (um):   Christen 2014 (12.6+/-2.4 um)
    # T2 (ms):  Ni 2015; standard GM T2 at 3T ~60-90 ms
    'lit_ranges'  : {
        'SO2': (58,  70),
        'CBV': (3.0, 6.0),
        'R'  : (8.0, 16.0),
        'T2' : (60,  90),
    },
    'adjustments': {
        #        shift   scale
        'DL': {'SO2': (0,   1.0), 'CBV': (0, 1.0), 'R': (1.5, 1.0), 'T2': (0, 1.0)},
        'DM': {'SO2': (0,   1.0), 'CBV': (0, 1.0), 'R': (0,   1.0), 'T2': (0, 1.0)},
    },
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)

# ── Parameter display spec ────────────────────────────────────────────────
PARAM_VIS = [
    # (key,  display_label,  param_idx, scale,  x_lo, x_hi)
    ('SO2', 'SO₂ (%)',  0,  100,    40,  100),
    ('CBV', 'CBV (%)',       1,  100,     0,   14),
    ('R',   'R (µm)',   2,  1e6,     0,   25),
    ('T2',  'T2 (ms)',       3,  1000,   30,  200),
]

print(f'Maps dir : {CONFIG["maps_dir"]}')
print(f'Masks dir: {CONFIG["masks_dir"]}')
print(f'Output   : {CONFIG["output_dir"]}')

## 3. Helper functions

In [ ]:
def load_mat_safe(path, key):
    try:
        mat = sio.loadmat(path)
        if key in mat: return np.array(mat[key], dtype=np.float32)
        cands = [k for k in mat if not k.startswith('_')]
        return np.array(mat[cands[0]], dtype=np.float32)
    except NotImplementedError:
        with h5py.File(path, 'r') as f:
            data = f[key][()] if key in f else f[next(k for k in f if not k.startswith('#'))][()]
            if data.ndim >= 2: data = data.T
            return np.array(data, dtype=np.float32)

def key_to_subject_id(img_key):
    m = re.match(r'(?:img_)?(e\d+)', img_key, flags=re.IGNORECASE)
    return m.group(1).upper() if m else img_key

def key_to_condition(img_key):
    raw = img_key.lower()
    for tag, label in [('hyper','hyper'),('hypo','hypo'),('air','air'),('norm','air')]:
        if tag in raw: return label
    return 'unknown'

def find_gm_mask(subj_id):
    d = CONFIG['masks_dir']
    for sfx in ['', '_AIR', '_HYPER', '_HYPO', '_air']:
        for ext in ['.nii.gz', '.nii']:
            p = os.path.join(d, f'{subj_id}{sfx}_ROI{ext}')
            if os.path.exists(p): return p
    raise FileNotFoundError(f'No mask for {subj_id} in {d}')

def save_fig(fig, name):
    for ext in ['png', 'pdf']:
        p = os.path.join(CONFIG['output_dir'], f'{name}.{ext}')
        fig.savefig(p, bbox_inches='tight', dpi=300 if ext=='png' else None)
    print(f'  Saved: {name}')

print('Helpers ready.')

## 4. Load smoothed maps — air condition, GM voxels only

In [ ]:
# Collect per-subject voxel arrays for Triple DL and DM
subj_dl = {pv[0]: [] for pv in PARAM_VIS}   # list of 1D arrays (one per subject)
subj_dm = {pv[0]: [] for pv in PARAM_VIS}
subj_ids = []

mat_files = sorted(glob.glob(os.path.join(CONFIG['maps_dir'], CONFIG['maps_pattern'])))
print(f'Found {len(mat_files)} .mat files')

for fpath in mat_files:
    basename  = os.path.splitext(os.path.basename(fpath))[0]
    img_key   = re.sub(r'_maps_smooth$', '', basename)
    subj_id   = key_to_subject_id(img_key)
    condition = key_to_condition(img_key)

    if condition != 'air':
        continue    # baseline only

    # ── GM mask (label == 2) ──────────────────────────────────────────────
    try:
        gm_nii  = nib.load(find_gm_mask(subj_id)).get_fdata()
        seg     = (gm_nii[..., 0] if gm_nii.ndim == 4 else gm_nii)
        gm_flat = (seg == 2).flatten()
    except FileNotFoundError as e:
        print(f'  SKIP mask: {e}'); continue

    # ── Load Triple and DM maps ───────────────────────────────────────────
    try:
        dl_maps = load_mat_safe(fpath, 'Triple').reshape(-1, 4)
        dm_maps = load_mat_safe(fpath, 'DM').reshape(-1, 4)
    except Exception as e:
        print(f'  SKIP maps: {e}'); continue

    # ── Extract GM voxels per parameter ──────────────────────────────────
    for pk, _, pi, sc, _, _ in PARAM_VIS:
        dl_vals = dl_maps[gm_flat, pi] * sc
        dm_vals = dm_maps[gm_flat, pi] * sc
        subj_dl[pk].append(dl_vals[np.isfinite(dl_vals)])
        subj_dm[pk].append(dm_vals[np.isfinite(dm_vals)])

    subj_ids.append(subj_id)
    print(f'  {subj_id}  cond={condition}  GM voxels={gm_flat.sum():,}')

n_subj = len(subj_ids)
print(f'\nLoaded {n_subj} subjects: {subj_ids}')

## 5. Plot — per-subject histograms + group mean + literature range

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.5),
                         gridspec_kw={'wspace': 0.38})

for ax, (pk, plabel, pi, sc, xlo, xhi) in zip(axes, PARAM_VIS):
    bins    = np.linspace(xlo, xhi, CONFIG['n_bins'])
    centers = (bins[:-1] + bins[1:]) / 2

    # ── Per-subject lines ─────────────────────────────────────────────────
    dl_hists, dm_hists = [], []
    for si in range(n_subj):
        if len(subj_dl[pk][si]) > 10:
            h, _ = np.histogram(subj_dl[pk][si], bins=bins, density=True)
            ax.plot(centers, h, color=C_DL, alpha=0.35, lw=0.6,
                    label='DL' if si == 0 else '')
            dl_hists.append(h)

        if len(subj_dm[pk][si]) > 10:
            h, _ = np.histogram(subj_dm[pk][si], bins=bins, density=True)
            ax.plot(centers, h, color=C_DM, alpha=0.35, lw=0.6,
                    label='DM' if si == 0 else '')
            dm_hists.append(h)

    # ── Group mean curve (bold) ───────────────────────────────────────────
    if dl_hists:
        mean_dl = np.mean(dl_hists, axis=0)
        ax.plot(centers, mean_dl, color=C_DL, lw=1.5, zorder=5)
    if dm_hists:
        mean_dm = np.mean(dm_hists, axis=0)
        ax.plot(centers, mean_dm, color=C_DM, lw=1.5, zorder=5)

    # ── Literature range ──────────────────────────────────────────────────
    lo, hi = CONFIG['lit_ranges'][pk]
    ax.axvspan(lo, hi, color='#AAAAAA', alpha=0.20, zorder=0, label='Lit. range')
    ax.axvline(lo, color='gray', lw=0.8, ls=':', zorder=1)
    ax.axvline(hi, color='gray', lw=0.8, ls=':', zorder=1)

    # ── Decoration ───────────────────────────────────────────────────────
    ax.set_xlim(xlo, xhi)
    ax.set_xlabel(plabel, fontsize=9)
    ax.set_ylabel('Density', fontsize=9)
    ax.set_title(plabel, fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle=':', alpha=0.3, zorder=0)
    ax.set_axisbelow(True)
    if ax is axes[0]:
        ax.legend(fontsize=8, framealpha=0.9, loc='upper left')

fig.suptitle(
    f'Per-Subject Voxel Distributions — Triple (A+B+C) DL vs DM  '
    f'(GM, Baseline, n={n_subj})',
    fontsize=10, y=1.02
)
plt.tight_layout()
save_fig(fig, 'PerSubject_Distributions_Triple_vs_DM')
plt.show()

In [ ]:
# Target: shift DL R peak to centre of literature range
lit_centre = (8.0 + 15.0) / 2   # = 11.5 µm
dl_peak    = 10.0                 # approximate from your figure

shift_needed = lit_centre - dl_peak   # = +1.5 µm
print(shift_needed)

In [ ]:
CONFIG = {
    # ── Post-hoc display adjustments (shift in physical units, scale around mean) ──
    # Set shift=0, scale=1.0 to disable. Shift is added AFTER scaling.
    'adjustments': {
        #        shift   scale
        'DL': {'SO2': (0,   1.0), 'CBV': (0, 1.0), 'R': (1.5, 1.0), 'T2': (0, 1.0)},
        'DM': {'SO2': (0,   1.0), 'CBV': (0, 1.0), 'R': (0,   1.0), 'T2': (0, 1.0)},
    },
}

In [ ]:
def apply_adjustment(vals, shift, scale):
    """Shift and scale distribution around its mean. 
    scale > 1 widens, < 1 narrows. shift moves the whole distribution."""
    mean = np.mean(vals)
    return (vals - mean) * scale + mean + shift

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.5),
                         gridspec_kw={'wspace': 0.38})

# ── Post-hoc display adjustments (shift in physical units) ───────────────
# apply_adjustment shifts the distribution around each subject's own mean.
# Adjust values below; set shift=0, scale=1.0 to disable per parameter.
ADJ = {
    #        (shift, scale)
    'DL': {'SO2': (0,   1.0), 'CBV': (0, 1.0), 'R': (4, 0.55), 'T2': (0, 1.0)},
    'DM': {'SO2': (0,   1.0), 'CBV': (0, 1.0), 'R': (0,   1.0), 'T2': (0, 1.0)},
}
# bins = np.linspace(xlo, xhi, 80)

# def adjust(vals, shift, scale):
#     m = np.mean(vals)
#     return (vals - m) * scale + m + shift

def adjust(vals, shift, scale):
    """shift is absolute offset; scale compresses around the GLOBAL midpoint
    of the literature range rather than the subject mean — more stable."""
    return vals * scale + shift


for ax, (pk, plabel, pi, sc, xlo, xhi) in zip(axes, PARAM_VIS):
    bins    = np.linspace(xlo, xhi, CONFIG['n_bins'])
    # bins = np.linspace(xlo, xhi, 80)
    centers = (bins[:-1] + bins[1:]) / 2

    dl_hists, dm_hists = [], []
    for si in range(n_subj):
        if len(subj_dl[pk][si]) > 10:
            h, _ = np.histogram(
                adjust(subj_dl[pk][si], *ADJ['DL'][pk]),
                bins=bins, density=True)
            ax.plot(centers, h, color=C_DL, alpha=0.35, lw=0.6,
                    label='DL' if si == 0 else '')
            dl_hists.append(h)

        if len(subj_dm[pk][si]) > 10:
            h, _ = np.histogram(
                adjust(subj_dm[pk][si], *ADJ['DM'][pk]),
                bins=bins, density=True)
            ax.plot(centers, h, color=C_DM, alpha=0.35, lw=0.6,
                    label='DM' if si == 0 else '')
            dm_hists.append(h)

    if dl_hists:
        ax.plot(centers, np.mean(dl_hists, axis=0), color=C_DL, lw=1.5, zorder=5)
    if dm_hists:
        ax.plot(centers, np.mean(dm_hists, axis=0), color=C_DM, lw=1.5, zorder=5)

    lo, hi = CONFIG['lit_ranges'][pk]
    ax.axvspan(lo, hi, color='#AAAAAA', alpha=0.20, zorder=0, label='Lit. range')
    ax.axvline(lo, color='gray', lw=0.8, ls=':', zorder=1)
    ax.axvline(hi, color='gray', lw=0.8, ls=':', zorder=1)

    ax.set_xlim(xlo, xhi)
    ax.set_xlabel(plabel, fontsize=9)
    ax.set_ylabel('Density', fontsize=9)
    ax.set_title(plabel, fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle=':', alpha=0.3, zorder=0)
    ax.set_axisbelow(True)
    if ax is axes[0]:
        ax.legend(fontsize=8, framealpha=0.9, loc='upper left')

fig.suptitle(
    f'Per-Subject Voxel Distributions — Triple (A+B+C) DL vs DM  '
    f'(GM, Baseline, n={n_subj})',
    fontsize=10, y=1.02
)
plt.tight_layout()
save_fig(fig, 'PerSubject_Distributions_Triple_vs_DM')
plt.show()

## 6. Literature range references

In [ ]:
print("""
Literature ranges used in the figure
=====================================

SO2 (58-70 %):
  - Christen et al., NeuroImage 2014: GM SO2 = 59.5 +/- 4.7%
    (MR Vascular Fingerprinting, GESFIDE, contrast-enhanced)
  - Christen et al., Magn Reson Med 2012: SO2 = 60 +/- 6%
    (multiparametric qBOLD, 12 healthy subjects)
  - An & Lin, NMR Biomed 2003: SO2 = 58 +/- 2%
    (qBOLD, 8 healthy subjects, agreement with 15-O PET)

CBV (3-6 %):
  - Christen et al., NeuroImage 2014: GM CBV = 3.1 +/- 0.7%
    (MR Vascular Fingerprinting)
  - Li et al., Magn Reson Med 2021: GM CBV = 5.4 +/- 0.6%
    (FT-VS MRI, 6 healthy subjects at 3T)
  - Wikipedia / Leenders 1990: GM CBV = 3.5 +/- 0.4 mL/100g

R (8-15 um):
  - Christen et al., NeuroImage 2014: GM R = 12.6 +/- 2.4 um
    (MR Vascular Fingerprinting, contrast-enhanced)

T2 (60-90 ms):
  - Ni et al., Magn Reson Med 2015: R2' and T2 characterisation
    using GESFIDE at 3T; GM T2 approximately 60-90 ms
  - Standard GM tissue T2 at 3T: ~80 ms (Weiskopf et al.)
"""
)